# FASE4-P4: Pipeline de Risco Pre-Entrega

**Objetivo:** Baseline logistico + XGBoost + SHAP + curva PR + score por vendedor  
**Referencia:** `docs/metrics_agreement.md` e `docs/feature_contract.md`  
**Ancora temporal:** `order_approved_at` — nenhuma feature pos-entrega no modelo

## (1) Load & Feature Matrix

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    precision_recall_curve,
)
from xgboost import XGBClassifier

sys.path.insert(0, "..")  # permite 'from src.features import ...' ao rodar do notebooks/
from src.features import PRE_DELIVERY_FEATURES, FORBIDDEN_FEATURES, TARGET_COLUMN

print(f"PRE_DELIVERY_FEATURES ({len(PRE_DELIVERY_FEATURES)} colunas): {PRE_DELIVERY_FEATURES}")
print(f"FORBIDDEN_FEATURES ({len(FORBIDDEN_FEATURES)} colunas): {FORBIDDEN_FEATURES}")
print(f"TARGET_COLUMN: {TARGET_COLUMN}")

In [ ]:
# Verificacao anti-leakage (FALHA RAPIDO se contrato violado)
leakage = [c for c in FORBIDDEN_FEATURES if c in PRE_DELIVERY_FEATURES]
assert not leakage, f"LEAKAGE DETECTADO: {leakage}"
print("OK: nenhum leakage em PRE_DELIVERY_FEATURES")

In [ ]:
df = pd.read_parquet("../data/gold/olist_gold.parquet")
print(f"Gold table: {len(df):,} linhas x {df.shape[1]} colunas")

missing = [c for c in PRE_DELIVERY_FEATURES if c not in df.columns]
assert not missing, f"Features ausentes na gold table: {missing}"
print(f"OK: todas as {len(PRE_DELIVERY_FEATURES)} features pre-entrega presentes")

In [ ]:
X = df[PRE_DELIVERY_FEATURES].copy()
y = df[TARGET_COLUMN].copy()

print(f"Dataset: {len(X):,} pedidos")
print(f"Classe positiva (bad_review=1): {y.mean():.1%}")
print(f"Features: {X.shape[1]} colunas")
print(f"\nNulos por feature:")
nulos = X.isnull().sum()[X.isnull().sum() > 0]
print(nulos if len(nulos) > 0 else "(nenhum nulo nas features)")

In [ ]:
NUMERIC_FEATURES = [
    "freight_value", "price", "freight_ratio", "estimated_delivery_days",
    "seller_customer_distance_km", "product_weight_g", "product_volume_cm3",
    "order_item_count", "payment_installments",
]
CATEGORICAL_FEATURES = [
    "seller_state", "customer_state",
    "product_category_name_english", "payment_type",
]

assert set(NUMERIC_FEATURES + CATEGORICAL_FEATURES) == set(PRE_DELIVERY_FEATURES), \
    "NUMERIC + CATEGORICAL nao cobrem exatamente PRE_DELIVERY_FEATURES"

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)
print(f"Treino: {len(X_train):,} | Teste: {len(X_test):,}")
print(f"Positivos treino: {y_train.mean():.1%} | Positivos teste: {y_test.mean():.1%}")

## (2) Baseline — Logistic Regression

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
])

baseline_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",  # LOCKED: sem SMOTE (decisao CONTEXT.md)
        max_iter=1000,
        random_state=42,
    )),
])

baseline_pipeline.fit(X_train, y_train)
print("Baseline pipeline treinado.")

In [ ]:
def eval_model(pipeline, X_test, y_test, name):
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    y_pred = pipeline.predict(X_test)
    pr_auc = average_precision_score(y_test, y_proba)
    print(f"\n{'='*40}")
    print(f"{name} | PR-AUC: {pr_auc:.4f}")
    print(classification_report(y_test, y_pred, target_names=["good_review", "bad_review"]))
    return pr_auc

baseline_pr_auc = eval_model(baseline_pipeline, X_test, y_test, "Baseline LogReg")

In [ ]:
import os
os.makedirs("../models", exist_ok=True)

joblib.dump(baseline_pipeline, "../models/baseline_logreg.joblib")
print("Salvo: models/baseline_logreg.joblib")

# Round-trip verification
loaded_baseline = joblib.load("../models/baseline_logreg.joblib")
sample_scores = loaded_baseline.predict_proba(X_test.head(3))[:, 1]
print(f"Round-trip OK: {sample_scores.round(3)}")

## (3) XGBoost

In [ ]:
# PLACEHOLDER — implementado no Plano 04-02
pass

## (4) SHAP

In [ ]:
# PLACEHOLDER — implementado no Plano 04-02
pass

## (5) Threshold + Estimativa Operacional

In [ ]:
# PLACEHOLDER — implementado no Plano 04-03
pass

## (6) Tabela de Vendedores

In [ ]:
# PLACEHOLDER — implementado no Plano 04-03
pass

## (7) Serializar .joblib

In [ ]:
# PLACEHOLDER — final_pipeline.joblib serializado no Plano 04-03
pass